# DeBERTa-NLI + mmBERT-base Fine-tune on MOOCCubeX CS

Notebook này làm đúng 2 stage bạn yêu cầu, không có GPT-5.4 trong loop:

1. **DeBERTa-v3-large-MNLI**: chấm direction `A -> B` vs `B -> A` cho edge bundle của mình.
2. **mmBERT-base fine-tune**: fine-tune binary prerequisite classifier trên **MOOCCubeX Computer Science prerequisites** rồi chấm strength cho edge bundle của mình.

Input bundle của project được tải từ **Google Drive link cũ**. Training data dùng **MOOCCubeX `prerequisites/cs.json`**. Notebook cũng cố tải thêm `entities/concept.json` để map concept id -> name/text; nếu không tải được thì vẫn cố parse trực tiếp từ `cs.json`.


In [ ]:
# Input bundle from the same Drive link as before
DRIVE_URL = "https://drive.google.com/file/d/1HyWTp9bJv2lg1EF17H_vxVSZnYLY4lDm/view?usp=sharing"
INPUT_PATH = "/content/edge_scoring_input_bundle_for_combo.json"

# Official-ish MOOCCubeX assets used for fine-tuning (CS only)
MOOCCUBEX_CS_URL = "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/prerequisites/cs.json"
MOOCCUBEX_CONCEPT_URL = "https://lfs.aminer.cn/misc/moocdata/data/mooccube2/entities/concept.json"
CS_JSON_PATH = "/content/mooccubex_cs.json"
CONCEPT_JSON_PATH = "/content/mooccubex_concept.json"

# Models
DEBERTA_MODEL_ID = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
MMBERT_MODEL_ID = "jhu-clsp/mmBERT-base"

# Runtime knobs
MAX_LENGTH_NLI = 512
MAX_LENGTH_FT = 256
NLI_BATCH_SIZE = 8
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
EPOCHS = 3
LR = 2e-4
WEIGHT_DECAY = 0.01
VAL_RATIO = 0.1
USE_LORA = True
SEED = 42
MAX_NEGATIVE_RATIO = 4
MAX_TRAIN_ROWS = 10000

# Decision thresholds for the combined output
NLI_MARGIN_THRESHOLD = 0.15
FT_STRONG_THRESHOLD = 0.60
FT_WEAK_THRESHOLD = 0.40
COMBO_MARGIN_THRESHOLD = 0.12

# Output
OUTPUT_PATH = "/content/deberta_mmbert_mooccubex_cs_scores.json"
SAVE_TO_DRIVE = False
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/deberta_mmbert_mooccubex_cs_scores.json"


In [ ]:
!pip -q install -U "transformers>=4.57.0" accelerate gdown safetensors sentencepiece protobuf peft scikit-learn requests

In [ ]:
import gc
import json
import math
import random
import re
import shutil
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

import requests
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from torch.utils.data import DataLoader, Dataset

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def extract_drive_file_id(url: str) -> str:
    patterns = [r"/file/d/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Cannot extract Google Drive file id from: {url}")


def download_from_drive(file_id: str, output_path: Path) -> None:
    import gdown

    result = gdown.download(id=file_id, output=str(output_path), quiet=False)
    if result and output_path.exists() and output_path.stat().st_size > 0:
        return

    session = requests.Session()
    url = "https://drive.google.com/uc?export=download"
    response = session.get(url, params={"id": file_id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break
    if token:
        response = session.get(url, params={"id": file_id, "confirm": token}, stream=True)
    response.raise_for_status()
    with output_path.open("wb") as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)


def download_http_file(url: str, output_path: Path, timeout: int = 120) -> None:
    response = requests.get(url, timeout=timeout, stream=True)
    response.raise_for_status()
    with output_path.open("wb") as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)


def load_json_flexible(path: Path) -> Any:
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Empty file: {path}")

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    rows = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            rows = []
            break
    if rows:
        return rows

    decoder = json.JSONDecoder()
    idx = 0
    objs = []
    length = len(text)
    while idx < length:
        while idx < length and text[idx].isspace():
            idx += 1
        if idx >= length:
            break
        obj, next_idx = decoder.raw_decode(text, idx)
        objs.append(obj)
        idx = next_idx
    if objs:
        return objs

    raise ValueError(f"Could not parse JSON/JSONL payload from {path}")


def load_json(path: Path) -> Any:
    return load_json_flexible(path)


def load_edge_bundle() -> dict[str, Any]:
    path = Path(INPUT_PATH)
    if path.exists() and path.stat().st_size > 0:
        print("Loading input bundle:", path)
        return load_json(path)

    file_id = extract_drive_file_id(DRIVE_URL)
    temp_download = Path("/content/edge_scoring_drive_download")
    if temp_download.exists():
        temp_download.unlink()
    download_from_drive(file_id, temp_download)
    print("Downloaded:", temp_download, "bytes=", temp_download.stat().st_size)

    if zipfile.is_zipfile(temp_download):
        extract_dir = Path("/content/edge_scoring_extract")
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(temp_download, "r") as zf:
            zf.extractall(extract_dir)
        candidates = list(extract_dir.rglob("*.json"))
        if not candidates:
            raise FileNotFoundError("Downloaded zip has no JSON file")
        chosen = None
        for candidate in candidates:
            if "edge_scoring_input_bundle_for_combo" in candidate.name or "edge_scoring_input_bundle" in candidate.name:
                chosen = candidate
                break
        chosen = chosen or candidates[0]
        shutil.copyfile(chosen, path)
        print("Extracted JSON:", chosen)
    else:
        shutil.copyfile(temp_download, path)

    return load_json(path)


def ensure_mooccubex_files() -> tuple[Path, Path | None]:
    cs_path = Path(CS_JSON_PATH)
    concept_path = Path(CONCEPT_JSON_PATH)
    if not cs_path.exists() or cs_path.stat().st_size == 0:
        print("Downloading MOOCCubeX CS prerequisites from:", MOOCCUBEX_CS_URL)
        download_http_file(MOOCCUBEX_CS_URL, cs_path)
    if not concept_path.exists() or concept_path.stat().st_size == 0:
        try:
            print("Downloading MOOCCubeX concept catalog from:", MOOCCUBEX_CONCEPT_URL)
            download_http_file(MOOCCUBEX_CONCEPT_URL, concept_path)
        except Exception as exc:
            print("Concept catalog download failed; will parse without it:", repr(exc))
            concept_path = None
    return cs_path, concept_path


In [ ]:
payload = load_edge_bundle()
clean_edges = payload.get("clean_candidate_edges") or []
pruned_edges = payload.get("pruned_edges") or []
global_kps = payload.get("global_kps") or payload.get("concepts_kp_global") or []
if not isinstance(clean_edges, list) or not isinstance(pruned_edges, list):
    raise ValueError("Expected clean_candidate_edges[] and pruned_edges[] in the edge bundle")

kp_index = {
    (kp.get("global_kp_id") or kp.get("kp_id")): kp
    for kp in global_kps
    if isinstance(kp, dict) and (kp.get("global_kp_id") or kp.get("kp_id"))
}


def readable_kp_id(kp_id: str) -> str:
    return re.sub(r"[_-]+", " ", kp_id.removeprefix("kp_")).strip()


def kp_text(kp_id: str) -> str:
    kp = kp_index.get(kp_id, {})
    name = kp.get("name") or kp.get("canonical_name") or readable_kp_id(kp_id)
    desc = kp.get("description") or kp.get("global_description") or ""
    return f"{name}. {desc}".strip()


def pair_text(edge: dict[str, Any], reverse: bool = False) -> tuple[str, str]:
    source_id = edge["target_kp_id"] if reverse else edge["source_kp_id"]
    target_id = edge["source_kp_id"] if reverse else edge["target_kp_id"]
    return kp_text(source_id), kp_text(target_id)


print("Clean edges:", len(clean_edges))
print("Pruned edges:", len(pruned_edges))
print("KP catalog entries:", len(kp_index))
print("Sample edge:", clean_edges[0] if clean_edges else None)


## Load and normalize MOOCCubeX CS prerequisite data

Mục tiêu ở đây là dựng supervised pairs cho `mmBERT-base` từ **MOOCCubeX Computer Science prerequisites**. Vì schema thực tế của `cs.json` có thể thay đổi, parser dưới đây cố xử lý theo hướng chịu lỗi:
- nếu `cs.json` đã có text trực tiếp thì dùng luôn,
- nếu `cs.json` chủ yếu chứa concept IDs thì cố map qua `entities/concept.json`,
- nếu tập labels chỉ có positive edges thì tự tạo hard negatives bằng **reverse edges + corrupted target sampling**.


In [ ]:
cs_path, concept_path = ensure_mooccubex_files()
raw_cs = load_json_flexible(cs_path)
print("CS prerequisite file:", cs_path, "type=", type(raw_cs).__name__)
if concept_path is not None and concept_path.exists():
    print("Concept catalog:", concept_path, "MB=", round(concept_path.stat().st_size / (1024 * 1024), 2))
else:
    print("Concept catalog: unavailable")


In [ ]:
def load_concept_rows(path: Path | None) -> list[dict[str, Any]]:
    if path is None or not path.exists():
        return []
    parsed = load_json_flexible(path)
    if isinstance(parsed, list):
        return [row for row in parsed if isinstance(row, dict)]
    if isinstance(parsed, dict):
        rows = []
        for value in parsed.values():
            if isinstance(value, list):
                rows.extend(row for row in value if isinstance(row, dict))
            elif isinstance(value, dict):
                rows.append(value)
        return rows
    return []


def first_nonempty(row: dict[str, Any], keys: list[str]) -> Any:
    for key in keys:
        if key in row and row[key] not in (None, "", [], {}):
            return row[key]
    return None


def normalize_label(value: Any) -> int | None:
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, int):
        return 1 if value > 0 else 0
    if isinstance(value, float):
        return 1 if value > 0 else 0
    if isinstance(value, str):
        lowered = value.strip().lower()
        if lowered in {"1", "true", "yes", "y", "positive", "entailment", "prerequisite", "is_prerequisite"}:
            return 1
        if lowered in {"0", "false", "no", "n", "negative", "contradiction", "not_prerequisite", "not prerequisite"}:
            return 0
    return None


def coerce_text(value: Any) -> str | None:
    if isinstance(value, str):
        value = value.strip()
        return value or None
    if isinstance(value, list):
        items = [coerce_text(item) for item in value]
        items = [item for item in items if item]
        return "; ".join(items) if items else None
    return None


concept_rows = load_concept_rows(concept_path)
concept_index: dict[str, dict[str, Any]] = {}
for row in concept_rows:
    cid = first_nonempty(row, ["id", "concept_id", "ccid", "cid", "_id"])
    if cid is None:
        continue
    cid = str(cid)
    concept_index[cid] = {
        "name": coerce_text(first_nonempty(row, ["name", "title", "concept", "concept_name", "entity"])) or cid,
        "description": coerce_text(first_nonempty(row, ["description", "desc", "content", "definition", "summary"])),
        "raw": row,
    }

print("Loaded concept rows:", len(concept_rows))
print("Concept index size:", len(concept_index))


In [ ]:
SOURCE_ID_KEYS = [
    "source_id", "source", "source_concept_id", "from", "head", "src", "pre_id", "concept1_id", "c1_id", "left_id", "prerequisite_id"
]
TARGET_ID_KEYS = [
    "target_id", "target", "target_concept_id", "to", "tail", "dst", "post_id", "concept2_id", "c2_id", "right_id", "dependent_id"
]
SOURCE_TEXT_KEYS = [
    "source_name", "source_text", "source_concept", "from_name", "head_name", "pre_name", "concept1", "c1", "left_name", "prerequisite"
]
TARGET_TEXT_KEYS = [
    "target_name", "target_text", "target_concept", "to_name", "tail_name", "post_name", "concept2", "c2", "right_name", "dependent"
]
LABEL_KEYS = [
    "label", "gold_label", "human_label", "is_prerequisite", "prereq", "prerequisite_label", "relation_label", "y", "ground_truth"
]
NODE_ID_KEYS = ["id", "concept_id", "ccid", "cid", "_id", "conceptId"]
NODE_TEXT_KEYS = ["name", "title", "concept", "concept_name", "entity", "text", "label", "display_name"]
PREREQ_LIST_KEYS = [
    "prerequisites", "prerequisite", "prerequisite_ids", "pre_concepts", "preconcepts", "parents", "dependencies", "requires", "require", "pres"
]
CHILD_LIST_KEYS = [
    "successors", "children", "dependents", "next", "targets", "postrequisites", "outputs"
]


def concept_text_from_id(cid: str | None) -> str | None:
    if cid is None:
        return None
    row = concept_index.get(str(cid))
    if not row:
        return None
    name = row["name"]
    desc = row.get("description")
    return f"{name}. {desc}".strip() if desc else name


def concept_text_from_ref(value: Any) -> str | None:
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return None
        return concept_text_from_id(value) or value
    if isinstance(value, (int, float)):
        return concept_text_from_id(str(value)) or str(value)
    if isinstance(value, dict):
        text = coerce_text(first_nonempty(value, NODE_TEXT_KEYS))
        if text:
            return text
        cid = first_nonempty(value, NODE_ID_KEYS)
        if cid is not None:
            return concept_text_from_id(str(cid)) or str(cid)
    return None


def normalize_pair_dict(row: dict[str, Any]) -> dict[str, Any] | None:
    label = None
    for key in LABEL_KEYS:
        if key in row:
            label = normalize_label(row.get(key))
            if label is not None:
                break

    source_id = first_nonempty(row, SOURCE_ID_KEYS)
    target_id = first_nonempty(row, TARGET_ID_KEYS)
    source_text = coerce_text(first_nonempty(row, SOURCE_TEXT_KEYS)) or concept_text_from_id(str(source_id))
    target_text = coerce_text(first_nonempty(row, TARGET_TEXT_KEYS)) or concept_text_from_id(str(target_id))

    if label is None:
        relation = coerce_text(first_nonempty(row, ["relation", "type", "rel"]))
        if relation:
            label = normalize_label(relation)

    if source_text and target_text and label is not None:
        return {
            "source_text": source_text,
            "target_text": target_text,
            "label": int(label),
            "source_id": str(source_id) if source_id is not None else None,
            "target_id": str(target_id) if target_id is not None else None,
            "raw": row,
        }

    return None


def normalize_pair_row(row: Any) -> list[dict[str, Any]]:
    if isinstance(row, dict):
        normalized = normalize_pair_dict(row)
        if normalized is not None:
            return [normalized]
        return adjacency_pairs_from_row(row)
    if isinstance(row, (list, tuple)) and len(row) >= 2:
        source_text = concept_text_from_ref(row[0])
        target_text = concept_text_from_ref(row[1])
        label = normalize_label(row[2]) if len(row) >= 3 else 1
        if source_text and target_text and label is not None:
            return [{
                "source_text": source_text,
                "target_text": target_text,
                "label": int(label),
                "source_id": None,
                "target_id": None,
                "raw": row,
            }]
    return []


def adjacency_pairs_from_row(row: dict[str, Any]) -> list[dict[str, Any]]:
    node_id = first_nonempty(row, NODE_ID_KEYS)
    node_text = concept_text_from_ref(first_nonempty(row, NODE_TEXT_KEYS)) or concept_text_from_id(str(node_id))
    if not node_text:
        return []

    out = []
    for key in PREREQ_LIST_KEYS:
        refs = row.get(key)
        if isinstance(refs, list):
            for ref in refs:
                prereq_text = concept_text_from_ref(ref)
                if prereq_text and prereq_text != node_text:
                    out.append({
                        "source_text": prereq_text,
                        "target_text": node_text,
                        "label": 1,
                        "source_id": None,
                        "target_id": str(node_id) if node_id is not None else None,
                        "raw": row,
                    })
    for key in CHILD_LIST_KEYS:
        refs = row.get(key)
        if isinstance(refs, list):
            for ref in refs:
                child_text = concept_text_from_ref(ref)
                if child_text and child_text != node_text:
                    out.append({
                        "source_text": node_text,
                        "target_text": child_text,
                        "label": 1,
                        "source_id": str(node_id) if node_id is not None else None,
                        "target_id": None,
                        "raw": row,
                    })
    return out


def walk_pairs(obj: Any) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    if isinstance(obj, list):
        for row in obj:
            out.extend(normalize_pair_row(row))
            if isinstance(row, dict):
                for value in row.values():
                    if isinstance(value, list) and value and isinstance(value[0], (dict, list, tuple)):
                        out.extend(walk_pairs(value))
    elif isinstance(obj, dict):
        direct = normalize_pair_dict(obj)
        if direct is not None:
            out.append(direct)
        out.extend(adjacency_pairs_from_row(obj))
        for value in obj.values():
            if isinstance(value, (list, dict)):
                out.extend(walk_pairs(value))
    return out

parsed_pairs = walk_pairs(raw_cs)
print('Parsed MOOCCubeX pair rows:', len(parsed_pairs))
if parsed_pairs:
    print('Sample parsed row:', parsed_pairs[0])
else:
    if isinstance(raw_cs, list) and raw_cs:
        print('Sample raw row:', raw_cs[0])
    elif isinstance(raw_cs, dict):
        print('Sample raw top-level keys:', list(raw_cs.keys())[:20])
        first_key = next(iter(raw_cs.keys()), None)
        if first_key is not None:
            print('Sample raw value:', raw_cs[first_key])


In [ ]:
if not parsed_pairs:
    raise RuntimeError(
        "Could not parse any supervised pairs from MOOCCubeX cs.json after schema-aware parsing. Inspect the sample raw row printed in the previous cell."
    )

# Deduplicate text pairs and keep the published negative labels, but downsample heavily so
# mmBERT-base fine-tuning stays practical on Colab and does not collapse into majority-class bias.
positive_pairs = set()
negative_pairs = set()
all_concepts = set()
for row in parsed_pairs:
    src = row["source_text"].strip()
    tgt = row["target_text"].strip()
    if not src or not tgt or src == tgt:
        continue
    all_concepts.add(src)
    all_concepts.add(tgt)
    if row["label"] == 1:
        positive_pairs.add((src, tgt))
    else:
        negative_pairs.add((src, tgt))

concept_list = sorted(all_concepts)
rng = random.Random(SEED)

if not negative_pairs:
    for src, tgt in list(positive_pairs):
        rev = (tgt, src)
        if rev not in positive_pairs:
            negative_pairs.add(rev)
    pos_list = list(positive_pairs)
    while len(negative_pairs) < len(positive_pairs) and concept_list:
        src, tgt = rng.choice(pos_list)
        corrupted_tgt = rng.choice(concept_list)
        candidate = (src, corrupted_tgt)
        if src != corrupted_tgt and candidate not in positive_pairs:
            negative_pairs.add(candidate)

max_negative_count = min(len(negative_pairs), max(len(positive_pairs), len(positive_pairs) * MAX_NEGATIVE_RATIO))
negative_pairs_list = sorted(negative_pairs)
rng.shuffle(negative_pairs_list)
negative_pairs = set(negative_pairs_list[:max_negative_count])

rows = [{"source_text": src, "target_text": tgt, "label": 1} for src, tgt in sorted(positive_pairs)]
rows += [{"source_text": src, "target_text": tgt, "label": 0} for src, tgt in sorted(negative_pairs)]
rng.shuffle(rows)

if len(rows) > MAX_TRAIN_ROWS:
    pos_rows_all = [row for row in rows if row['label'] == 1]
    neg_rows_all = [row for row in rows if row['label'] == 0]
    kept_pos = pos_rows_all[: min(len(pos_rows_all), MAX_TRAIN_ROWS // (MAX_NEGATIVE_RATIO + 1) or len(pos_rows_all))]
    remaining = max(0, MAX_TRAIN_ROWS - len(kept_pos))
    kept_neg = neg_rows_all[: min(len(neg_rows_all), remaining)]
    rows = kept_pos + kept_neg
    rng.shuffle(rows)

pos_rows = [row for row in rows if row['label'] == 1]
neg_rows = [row for row in rows if row['label'] == 0]
val_pos = max(1, int(len(pos_rows) * VAL_RATIO)) if len(pos_rows) > 3 else 1
val_neg = max(1, int(len(neg_rows) * VAL_RATIO)) if len(neg_rows) > 3 else 1
val_rows = pos_rows[:val_pos] + neg_rows[:val_neg]
train_rows = pos_rows[val_pos:] + neg_rows[val_neg:]
rng.shuffle(train_rows)
rng.shuffle(val_rows)

print({
    'all_rows_after_sampling': len(rows),
    'train_rows': len(train_rows),
    'val_rows': len(val_rows),
    'positive': len(pos_rows),
    'negative': len(neg_rows),
    'max_negative_ratio': MAX_NEGATIVE_RATIO,
})
print('train label counts', Counter(row['label'] for row in train_rows))
print('val label counts', Counter(row['label'] for row in val_rows))


## Stage 1: DeBERTa-NLI direction scoring on the project edge bundle

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device=", device, "dtype=", dtype)
if device == "cuda":
    print("gpu=", torch.cuda.get_device_name(0))

deb_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_MODEL_ID)
deb_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_MODEL_ID, torch_dtype=dtype).to(device)
deb_model.eval()

label_to_id = {str(v).lower(): int(k) for k, v in deb_model.config.id2label.items()}
entailment_id = next((i for lab, i in label_to_id.items() if "entail" in lab), None)
contradiction_id = next((i for lab, i in label_to_id.items() if "contrad" in lab), None)
neutral_id = next((i for lab, i in label_to_id.items() if "neutral" in lab), None)
if entailment_id is None or contradiction_id is None:
    raise RuntimeError(f"Cannot find entailment/contradiction labels in {deb_model.config.id2label}")
print("id2label=", deb_model.config.id2label)


def nli_pair(edge: dict[str, Any], reverse: bool = False) -> tuple[str, str]:
    source_text, target_text = pair_text(edge, reverse=reverse)
    premise = f"Source concept: {source_text}\nTarget concept: {target_text}"
    hypothesis = "Understanding the source concept is required before understanding the target concept."
    return premise, hypothesis


def score_nli_pairs(pairs: list[tuple[str, str]], batch_size: int = NLI_BATCH_SIZE) -> list[dict[str, float]]:
    out = []
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        premises = [p for p, h in batch]
        hypotheses = [h for p, h in batch]
        encoded = deb_tokenizer(
            premises,
            hypotheses,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH_NLI,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            logits = deb_model(**encoded).logits.float()
        probs = F.softmax(logits, dim=-1)
        for row in range(probs.shape[0]):
            out.append({
                'entailment': probs[row, entailment_id].item(),
                'contradiction': probs[row, contradiction_id].item(),
                'neutral': probs[row, neutral_id].item() if neutral_id is not None else 0.0,
            })
    return out

forward_nli = score_nli_pairs([nli_pair(edge, reverse=False) for edge in clean_edges])
reverse_nli = score_nli_pairs([nli_pair(edge, reverse=True) for edge in clean_edges])
print('Scored NLI edges:', len(forward_nli))


## Stage 2: Fine-tune mmBERT-base on MOOCCubeX CS

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

class PairDataset(Dataset):
    def __init__(self, rows, tokenizer, max_length):
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        encoded = self.tokenizer(
            row['source_text'],
            row['target_text'],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt',
        )
        item = {k: v.squeeze(0) for k, v in encoded.items()}
        item['labels'] = torch.tensor(row['label'], dtype=torch.long)
        return item


def collate_fn(batch):
    keys = batch[0].keys()
    return {key: torch.stack([row[key] for row in batch]) for key in keys}

mb_tokenizer = AutoTokenizer.from_pretrained(MMBERT_MODEL_ID)
mb_model = AutoModelForSequenceClassification.from_pretrained(
    MMBERT_MODEL_ID,
    num_labels=2,
    torch_dtype=dtype,
    attn_implementation='sdpa',
).to(device)

peft_enabled = False
if USE_LORA:
    try:
        from peft import LoraConfig, TaskType, get_peft_model
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules='all-linear',
        )
        mb_model = get_peft_model(mb_model, lora_config)
        peft_enabled = True
        mb_model.print_trainable_parameters()
    except Exception as exc:
        print('LoRA attach failed; fallback to classifier tuning:', repr(exc))
        for name, param in mb_model.named_parameters():
            param.requires_grad = name.startswith('classifier') or 'score' in name

train_ds = PairDataset(train_rows, mb_tokenizer, MAX_LENGTH_FT)
val_ds = PairDataset(val_rows, mb_tokenizer, MAX_LENGTH_FT)
train_loader = DataLoader(train_ds, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optim_params = [param for param in mb_model.parameters() if param.requires_grad]
optimizer = torch.optim.AdamW(optim_params, lr=LR, weight_decay=WEIGHT_DECAY)
num_train_steps = max(1, math.ceil(len(train_loader) / GRAD_ACCUM_STEPS) * EPOCHS)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(0.1 * num_train_steps)),
    num_training_steps=num_train_steps,
)

print('train batches', len(train_loader), 'val batches', len(val_loader), 'num_train_steps', num_train_steps)


In [ ]:
def run_eval(model, loader):
    model.eval()
    all_labels = []
    all_preds = []
    all_probs = []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            output = model(**batch)
            total_loss += output.loss.item()
            probs = F.softmax(output.logits.float(), dim=-1)[:, 1]
            preds = (probs >= 0.5).long()
            all_probs.extend(probs.detach().cpu().tolist())
            all_preds.extend(preds.detach().cpu().tolist())
            all_labels.extend(batch['labels'].detach().cpu().tolist())
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary', zero_division=0)
    return {
        'loss': total_loss / max(1, len(loader)),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

best_state = None
best_f1 = -1.0
history = []
global_step = 0

for epoch in range(EPOCHS):
    mb_model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}
        output = mb_model(**batch)
        loss = output.loss / GRAD_ACCUM_STEPS
        loss.backward()
        running_loss += loss.item() * GRAD_ACCUM_STEPS

        if step % GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(optim_params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

    metrics = run_eval(mb_model, val_loader)
    metrics['epoch'] = epoch + 1
    metrics['train_loss'] = running_loss / max(1, len(train_loader))
    history.append(metrics)
    print(metrics)

    if metrics['f1'] > best_f1:
        best_f1 = metrics['f1']
        if peft_enabled:
            best_state = {k: v.detach().cpu() for k, v in mb_model.state_dict().items() if 'lora_' in k or 'classifier' in k or 'score' in k}
        else:
            best_state = {k: v.detach().cpu() for k, v in mb_model.state_dict().items() if v.requires_grad or k.startswith('classifier')}

if best_state is not None:
    current = mb_model.state_dict()
    current.update(best_state)
    mb_model.load_state_dict(current, strict=False)

print('best_val_f1', best_f1)


## Stage 3: Score project edges with the fine-tuned mmBERT-base

In [ ]:
def score_strength_rows(rows: list[dict[str, str]], batch_size: int = EVAL_BATCH_SIZE) -> list[float]:
    mb_model.eval()
    out = []
    for start in range(0, len(rows), batch_size):
        batch_rows = rows[start:start + batch_size]
        encoded = mb_tokenizer(
            [row['source_text'] for row in batch_rows],
            [row['target_text'] for row in batch_rows],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH_FT,
            return_tensors='pt',
        ).to(device)
        with torch.no_grad():
            logits = mb_model(**encoded).logits.float()
        probs = F.softmax(logits, dim=-1)[:, 1]
        out.extend(probs.detach().cpu().tolist())
    return out

forward_strength_rows = []
reverse_strength_rows = []
for edge in clean_edges:
    src_text, tgt_text = pair_text(edge, reverse=False)
    forward_strength_rows.append({'source_text': src_text, 'target_text': tgt_text})
    rev_src_text, rev_tgt_text = pair_text(edge, reverse=True)
    reverse_strength_rows.append({'source_text': rev_src_text, 'target_text': rev_tgt_text})

forward_strength = score_strength_rows(forward_strength_rows)
reverse_strength = score_strength_rows(reverse_strength_rows)
print('Scored strength edges:', len(forward_strength))


## Stage 4: Combine NLI direction + fine-tuned strength into one output artifact

In [ ]:
def direction_confidence_from_margin(margin: float) -> str:
    abs_margin = abs(margin)
    if abs_margin >= 0.30:
        return 'high'
    if abs_margin >= 0.15:
        return 'medium'
    return 'low'


def combo_decision(nli_margin: float, ft_forward: float, ft_reverse: float) -> tuple[str, str]:
    ft_margin = ft_forward - ft_reverse
    combo_margin = 0.6 * nli_margin + 0.4 * ft_margin
    best_strength = max(ft_forward, ft_reverse)

    if combo_margin >= COMBO_MARGIN_THRESHOLD and ft_forward >= FT_STRONG_THRESHOLD:
        return 'keep', direction_confidence_from_margin(combo_margin)
    if combo_margin <= -COMBO_MARGIN_THRESHOLD and ft_reverse >= FT_STRONG_THRESHOLD:
        return 'flip', direction_confidence_from_margin(combo_margin)
    if best_strength < FT_WEAK_THRESHOLD:
        return 'prune', 'low'
    return 'defer', direction_confidence_from_margin(combo_margin)

edge_scores = []
for idx, edge in enumerate(clean_edges):
    nli_f = forward_nli[idx]
    nli_r = reverse_nli[idx]
    nli_margin = nli_f['entailment'] - nli_r['entailment']
    ft_f = float(forward_strength[idx])
    ft_r = float(reverse_strength[idx])
    ft_margin = ft_f - ft_r
    decision, decision_conf = combo_decision(nli_margin, ft_f, ft_r)

    edge_scores.append({
        'source_kp_id': edge['source_kp_id'],
        'target_kp_id': edge['target_kp_id'],
        'deberta_forward_entailment': nli_f['entailment'],
        'deberta_forward_contradiction': nli_f['contradiction'],
        'deberta_reverse_entailment': nli_r['entailment'],
        'deberta_reverse_contradiction': nli_r['contradiction'],
        'deberta_direction_margin': nli_margin,
        'deberta_direction_confidence': direction_confidence_from_margin(nli_margin),
        'mmbert_forward_prereq_probability': ft_f,
        'mmbert_reverse_prereq_probability': ft_r,
        'mmbert_strength_margin': ft_margin,
        'combo_decision': decision,
        'combo_decision_confidence': decision_conf,
        'input_edge_scope': edge.get('edge_scope'),
        'input_keep_confidence': edge.get('keep_confidence'),
        'input_review_status': edge.get('review_status'),
    })

summary = Counter(row['combo_decision'] for row in edge_scores)
print('decision_summary', dict(summary))
print('sample', json.dumps(edge_scores[0], ensure_ascii=False, indent=2)[:1200] if edge_scores else 'NONE')


In [ ]:
output_payload = {
    'input_source': {
        'drive_url': DRIVE_URL,
        'input_path': INPUT_PATH,
        'clean_candidate_edges': len(clean_edges),
        'pruned_edges': len(pruned_edges),
        'global_kps': len(global_kps),
    },
    'training_source': {
        'dataset': 'MOOCCubeX prerequisite CS',
        'cs_url': MOOCCUBEX_CS_URL,
        'concept_url': MOOCCUBEX_CONCEPT_URL,
        'train_rows': len(train_rows),
        'val_rows': len(val_rows),
        'label_distribution_train': dict(Counter(row['label'] for row in train_rows)),
        'label_distribution_val': dict(Counter(row['label'] for row in val_rows)),
    },
    'models': {
        'deberta_nli_model_id': DEBERTA_MODEL_ID,
        'mmbert_model_id': MMBERT_MODEL_ID,
        'mmbert_peft_enabled': peft_enabled,
    },
    'mmbert_training_history': history,
    'edge_scores': edge_scores,
}

Path(OUTPUT_PATH).write_text(json.dumps(output_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Wrote', OUTPUT_PATH)
print('Output MB', round(Path(OUTPUT_PATH).stat().st_size / (1024 * 1024), 2))


In [ ]:
# Optional save to Drive + optional browser download
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        shutil.copyfile(OUTPUT_PATH, DRIVE_OUTPUT_PATH)
        print('Saved to Drive:', DRIVE_OUTPUT_PATH)
    except Exception as exc:
        print('Drive save skipped:', repr(exc))

try:
    from google.colab import files
    files.download(OUTPUT_PATH)
except Exception as exc:
    print('Download UI unavailable in this environment:', repr(exc))
